# Lab 2: Core Definitions, Degree Sequences, and Isomorphism

In this lab, we combine the foundational work of Module 2 into one notebook.

We will move through four connected themes:

- core graph definitions and representations,
- connectedness, paths, cycles, and bipartite structure,
- degree tables, degree sequences, and degree histograms,
- graph isomorphism and what degree information can and cannot tell us.

The goal is not just to run code, but to connect the computations back to the mathematical definitions from lecture.

In [ ]:
import networkx as nx
from labs.module_2.lab2_helpers import (
    build_graph_from_edge_list,
    adjacency_dict,
    basic_summary,
    graph_summary,
    degree_table,
    degree_histogram_data,
    plot_degree_histogram,
    draw_graph,
    export_for_gephi,
    path_between,
    cycle_basis,
    induced_subgraph,
    degree_sequence,
    isomorphic,
    isomorphism_mapping,
    relabel_randomly,
    generate_random_graph,
    time_isomorphism_check,
    plot_isomorphism_timings,
)

### Part 1: Build a graph and inspect its basic structure

We begin with a small graph and inspect its edge list, adjacency structure, and summary statistics.

In [ ]:
edges = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("C", "D"),
    ("D", "E"),
    ("E", "F"),
    ("G", "H"),
]

G1 = build_graph_from_edge_list(edges)
basic_summary(G1)

In [ ]:
adjacency_dict(edges)

In [ ]:
graph_summary(G1)

In [ ]:
draw_graph(G1, _title="Original graph G1")

Questions:
- How many vertices and edges does the graph have?
- How many connected components does it have?
- Which vertices lie in the same component?
- What does the adjacency dictionary make easy to see?

### Part 2: Connectedness, paths, and components

Now modify the graph so that it becomes connected, then explore paths in the new graph.

In [ ]:
edges_connected = edges + [("F", "G")]
G2 = build_graph_from_edge_list(edges_connected)

graph_summary(G2)

In [ ]:
draw_graph(G2, _title="Connected graph G2")

In [ ]:
path_between(G2, "A", "H")

In [ ]:
path_between(G2, "B", "E")

Questions:
- What edge was enough to connect the graph?
- Once the graph is connected, what changes about path existence?
- Why is path existence tied to connectedness?

### Part 3: Cycles, induced subgraphs, and bipartite structure

We now look for cycles and compare full graphs to induced subgraphs.

In [ ]:
cycle_basis(G2)

In [ ]:
H = induced_subgraph(G2, ["A", "B", "C", "D"])
graph_summary(H)

In [ ]:
draw_graph(H, _title="Induced subgraph on A, B, C, D")

In [ ]:
nx.is_bipartite(G1), nx.is_bipartite(G2), nx.is_bipartite(H)

Questions:
- Which graphs in this section contain cycles?
- Why is the induced subgraph on A, B, C, D different from the full graph?
- Why does a triangle obstruct bipartiteness?
- Can you build a connected graph that is bipartite?

### Part 4: Degree tables and degree histograms

Now degree becomes our main invariant.

We start locally with the degree of each vertex, then move to a global view using a histogram.

In [ ]:
degree_table(G2)

In [ ]:
degree_histogram_data(G2)

In [ ]:
plot_degree_histogram(G2, "Degree histogram for G2")

Questions:
- Which vertices have the largest degree?
- Does the graph appear to have a hub?
- What does the histogram reveal that the picture does not show as clearly?

### Part 5: Degree sequences as graph invariants

The degree sequence is a useful invariant for comparing graphs.

If two graphs have different degree sequences, then they cannot be isomorphic.
If two graphs have the same degree sequence, they still may or may not be isomorphic.

In [ ]:
edges_a = [
    ("A", "B"),
    ("A", "C"),
    ("A", "D"),
    ("B", "C"),
    ("B", "E"),
    ("C", "F"),
    ("D", "E"),
    ("D", "F"),
    ("E", "G"),
    ("F", "H"),
    ("G", "H"),
]

edges_b = [
    (1, 2),
    (1, 3),
    (1, 4),
    (2, 3),
    (2, 5),
    (3, 6),
    (4, 5),
    (4, 6),
    (5, 7),
    (6, 8),
    (7, 8),
]

edges_c = [
    ("A", "B"),
    ("A", "C"),
    ("A", "D"),
    ("B", "C"),
    ("B", "E"),
    ("C", "F"),
    ("D", "E"),
    ("D", "F"),
    ("E", "G"),
    ("F", "G"),
    ("G", "H"),
]

GA = build_graph_from_edge_list(edges_a)
GB = build_graph_from_edge_list(edges_b)
GC = build_graph_from_edge_list(edges_c)

In [ ]:
degree_sequence(GA), degree_sequence(GB), degree_sequence(GC)

In [ ]:
plot_degree_histogram(GA, "Degree histogram for GA")
plot_degree_histogram(GB, "Degree histogram for GB")
plot_degree_histogram(GC, "Degree histogram for GC")

Questions:
- Which of the degree sequences match?
- Which mismatch?
- What can a mismatch prove immediately?
- Why does a match fail to prove isomorphism by itself?

### Part 6: Isomorphism checking

Now use NetworkX to check which graphs are actually isomorphic.

In [ ]:
isomorphic(GA, GB)

In [ ]:
isomorphic(GA, GC)

In [ ]:
isomorphic(GB, GC)

In [ ]:
isomorphism_mapping(GA, GB)

Questions:
- Which pair is actually isomorphic?
- How does the mapping explain the similarity?
- Why is isomorphism about structure rather than labels?
- What information did the degree sequence get right, and what did it fail to decide?

### Part 7: Gephi comparison

Export the graphs and size the nodes by degree in Gephi.

In [ ]:
export_for_gephi(GA, "lab02_GA.gexf")
export_for_gephi(GB, "lab02_GB.gexf")
export_for_gephi(GC, "lab02_GC.gexf")

In Gephi:

1. Import the graphs.
2. Apply a layout.
3. Size nodes by degree.
4. Compare the visual structure of the graphs.

Questions:
- Do the two isomorphic graphs still look similar after relabeling?
- Can Gephi make degree patterns easier to see?
- Does visual similarity prove isomorphism?

### Part 8: Scaling the isomorphism check

Finally, test how the isomorphism checker behaves on somewhat larger random graphs.

This section is exploratory. The point is not to prove a complexity theorem, but to observe that computation time can grow as graph size grows.

In [ ]:
run_time = []
is_iso = []

n_verts = [40, 60, 80, 100, 120, 140, 160, 180, 200]
n_edges = [2 * n for n in n_verts]

for n, m in zip(n_verts, n_edges):
    big_graph_1 = generate_random_graph(n, m, seed=17)
    big_graph_2 = generate_random_graph(n, m, seed=19)
    result = time_isomorphism_check(big_graph_1, big_graph_2)
    run_time.append(result["time_seconds"])
    is_iso.append(result["isomorphic"])

run_time, is_iso

In [ ]:
plot_isomorphism_timings(n_verts, n_edges, run_time)

Questions:
- How does run time change as the graphs get larger?
- Why is it useful to separate graph invariants from full isomorphism checking?
- Why is timing evidence not the same thing as a proof about asymptotic complexity?

### Part 9: Reflection

Answer the following in complete sentences.

1. What is the difference between an edge list and an adjacency dictionary?
2. What does it mean for a graph to be connected?
3. What is an induced subgraph?
4. Why does a triangle prevent bipartiteness?
5. What is a degree sequence?
6. Why is the degree sequence a useful invariant?
7. Why is the degree sequence not sufficient to prove isomorphism?
8. What is graph isomorphism really comparing?
9. What observation from this lab most strongly suggests a theorem?